# GE2340 smoking detection: evaluation on Colab
Runtime -> Change runtime type -> **T4 GPU** before running.
Run cells top to bottom. Clips stay in Google Drive (never in the public repo).

In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Mount Drive: clips go in MyDrive/ge2340/clips
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Code + packages (no -U: keep Colab's CUDA build of torch)
!git clone https://github.com/YOUR_GITHUB_USERNAME/ge2340-smoking-detection.git
%cd ge2340-smoking-detection
!pip install -q "ultralytics>=8.4.0" pandas matplotlib huggingface_hub

In [ ]:
# 2. SAM 3 package. --no-deps avoids SAM 3's numpy<2 pin, which has no wheel for Colab's Python.
!pip install -q --no-deps git+https://github.com/facebookresearch/sam3.git
!pip install -q timm ftfy regex iopath einops psutil "setuptools<81"

In [ ]:
# 3. SAM 3 weights (mirror of Meta's official checkpoint; cite facebook/sam3 in the report)
from huggingface_hub import hf_hub_download
import shutil
shutil.copy(hf_hub_download('1038lab/sam3', 'sam3.pt'), 'models/sam3.pt')

In [ ]:
# 4. Smoke test: SAM 3 loads and scores one image
import sys, cv2, time; sys.path.insert(0, 'src')
from stage2_sam3 import Sam3Verifier
sam3 = Sam3Verifier('models/sam3.pt')
img = cv2.imread('/content/drive/MyDrive/ge2340/test.jpg')  # any photo of a person holding a cigarette
t = time.time(); print(sam3.score(img)[0], f'{(time.time()-t)*1000:.0f} ms')

In [ ]:
# 5. One GPU pass over all clips -> results/raw_scores.csv
!python src/run_eval.py --labels data/labels.csv --clips /content/drive/MyDrive/ge2340/clips --save-crops
!cp results/raw_scores.csv /content/drive/MyDrive/ge2340/

In [ ]:
# 6. Choose thresholds on DEV only, then freeze them
!python src/analyze.py --freeze
!cat results/frozen_thresholds.json

In [ ]:
# 7. Test-set tables and figures (uses the frozen thresholds)
!python src/analyze.py
!cp -r results /content/drive/MyDrive/ge2340/results_final